# Hatteras Island CASCADE Hindcast

Author: Hannah A. Henry, Coastal Environmental Change Lab, University of North Carolina at Chapel Hill
Contact: hahenry@unc.edu
Version: 2026-10-23

One run walked through step by step: run each section, read its report, look at its QC plot, and confirm it
is right before moving on. The run is chosen in `hat_run.yaml`.

Each section calls the same step `HAT_hindcast.py` runs: `setup_*` in `hindcast/HAT_hindcast_setup.py` before the
model is built, `pre_run_diagnostics`, `verify_run` and `write_run_outputs` in `hindcast/HAT_hindcast_outputs.py`
after, and `make_run_figures` (`figure_making/model_output/run_figures.py`) for the figures. Every value
a step builds goes on `r` (`r.START_YEAR`, `r.ELEVATION_FILE_PATHS`, ...). What is only here is the
walkthrough: the QC plots and their helpers.

## 1. Imports

In [ ]:
import base64
import os
import sys
import time
from pathlib import Path

# cascade_pipeline and hatteras_site_config live in Hatteras/ (the folder holding
# hatteras_paths.py), which isn't installed; data/ and output/ live in the workspace on D:
_here = Path.cwd().resolve()
SCRIPTS_DIR = next((p for p in (_here, *_here.parents) if (p / "hatteras_paths.py").exists()), None)
if SCRIPTS_DIR is None:
    raise RuntimeError(f"hatteras_paths.py not found above {_here}")
# The settings and the run's steps sit in hatteras_ms/hindcast/; the figures in figure_making/model_output/
for _path in (SCRIPTS_DIR, SCRIPTS_DIR / "hatteras_ms" / "hindcast",
              SCRIPTS_DIR / "figure_making" / "model_output"):
    if str(_path) not in sys.path:
        sys.path.insert(0, str(_path))

# --- the settings file, read once, before anything it selects ----------------
# hat_run.yaml chooses the run: edit it, save it, then Restart & Run All. Two of its values are settled
# at import -- use_sandbox_cascade picks the Cascade class, show_figures the matplotlib backend -- so an
# edit needs a fresh kernel rather than a re-run cell.
from HAT_hindcast_config import load_run_config

RUN_CONFIG = load_run_config()
os.environ["CASCADE_USE_SANDBOX"] = "1" if RUN_CONFIG.use_sandbox_cascade else "0"

# `output.show_figures: null` means "whichever file is being run decides": inline here, not in the .py
SHOW_FIGURES = True if RUN_CONFIG.show_figures is None else RUN_CONFIG.show_figures
import matplotlib
if not SHOW_FIGURES:
    matplotlib.use("Agg")

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display

# Fixes to obx-cascade's cascade/ that the lab has not merged yet, swapped in before any model is built
import cascade_patch
CASCADE_PATCHES = cascade_patch.apply()

from cascade_pipeline import roadway
from cascade_pipeline.annotations import add_geographic_annotations
from cascade_pipeline.hindcast import (
    DAM_TO_M,
    USE_SANDBOX_CASCADE,
    build_background_erosion,
    build_cascade,
    build_island_offset,
    groin_trapping_schedule,
    run_cascade_simulation,
)
from cascade_pipeline.plotting import init_planview, setup_qc
from cascade_pipeline.plotting.rate_comparison import DEFAULT_RATE_COMPARISON
from cascade_pipeline.plotting.road_planview import plot_roadway_planview
from cascade_pipeline.run_registry import guard_run_dir, run_dir_contents
from site_layer.hatteras_site_config import (
    HATTERAS_ANNOTATIONS,
    HATTERAS_BE_PRESETS,
    HATTERAS_BEACH_DUNE,
    HATTERAS_DOMAINS,
    HATTERAS_GEOMETRY,
    HATTERAS_PERIODS,
    HATTERAS_RELOCATION_CHECK_2004,
    HATTERAS_ROAD_EVENTS,
)

# The run's steps, the same ones HAT_hindcast.py calls
from HAT_hindcast_setup import (
    RunSetup,
    setup_topography,
    setup_scenario,
    setup_island_offset,
    setup_forcings,
    setup_roadway,
    setup_beach_dune,
    setup_groin,
    setup_run_name,
    setup_coastsat_target,
    setup_conventions,
    setup_parameters,
)
from HAT_hindcast_outputs import pre_run_diagnostics, verify_run, write_run_outputs
from run_figures import make_run_figures

# Everything the run builds goes on r, one attribute per value
r = RunSetup(RUN_CONFIG=RUN_CONFIG, CASCADE_PATCHES=CASCADE_PATCHES,
             GENERATED_BY="HAT_hindcast.ipynb")

print(f"Imports OK from {SCRIPTS_DIR}")
print(f"SHOW_FIGURES = {SHOW_FIGURES}")
print(f"USE_SANDBOX_CASCADE = {USE_SANDBOX_CASCADE}")
print(f"HATTERAS_DOMAINS.total_domains = {HATTERAS_DOMAINS.total_domains}  "
      f"(geometry {HATTERAS_GEOMETRY!r}: GIS {HATTERAS_DOMAINS.first_gis_id} "
      f"to {HATTERAS_DOMAINS.last_gis_id})")

## 2. Fixed dune/topo (period-independent) — QC plot
Same 2009 init surface for both periods, doesn't depend on `START_YEAR`.

Broken into steps so each can be confirmed before moving on:

| Step | What it does | What to check |
| --- | --- | --- |
| 2.1 | Resolve project paths | printed dirs all say `ok` |
| 2.2 | Build the 120 padded file paths | buffer/real/buffer order, 120 each |
| 2.3 | Verify every file exists | no exception |
| 2.4 | Units check vs Barrier3D contract | every check reports 90/90 |
| 2.5 | Browse per-domain extractor figures | dune picks and water trim, domain by domain |

### 2.1 Project paths

Everything derives from `SCRIPTS_DIR` (Section 1), so no absolute paths are
hardcoded. Layout under `data/hatteras_init/1-barrier3d-domains/`:

> Why this is set up the way it is: [HAT_hindcast_methods.md](docs/HAT_hindcast_methods.md#21-project-paths)

In [ ]:
# 2.1-2.4: the topography product and version, the project paths, the padded file lists, a check that
# every file exists, and the units check against Barrier3D's contract
setup_topography(r)

### 2.2 Build the padded file lists

CASCADE takes one elevation file and one dune file per padded domain, in
alongshore order: 15 buffers, then real GIS 1–90, then 15 buffers (120 total).
Both buffer ends reuse the same sample profile, so only the real domains vary.

The returned lists are index-aligned with the padded array, so
`HATTERAS_DOMAINS.gis_to_pad(gis_id)` indexes straight into them.

In [ ]:
_first_real = HATTERAS_DOMAINS.start_real_index
_last_real = HATTERAS_DOMAINS.end_real_index - 1

print("Padded array boundaries:")
for _label, _pad in [
    ("pad 0 (buffer)", 0),
    (f"pad {_first_real} (GIS {HATTERAS_DOMAINS.first_gis_id})", _first_real),
    (f"pad {_last_real} (GIS {HATTERAS_DOMAINS.last_gis_id})", _last_real),
    (f"pad {HATTERAS_DOMAINS.total_domains - 1} (buffer)",
     HATTERAS_DOMAINS.total_domains - 1),
]:
    print(f"  {_label:<22} {Path(r.ELEVATION_FILE_PATHS[_pad]).name}")

### 2.3 Verify every file exists

Checks all 240 paths up front. A stale `TOPO_DUNE_VERSION` or a moved data
folder fails here with a count and the first offender, rather than surfacing
as an empty plot or an opaque traceback inside Barrier3D's init.

Done by `setup_topography` above, which raises before anything else runs.

### 2.4 Units check against Barrier3D's input contract

Barrier3D's `load_input.py` converts the *scalar* YAML parameters but loads the
elevation and dune `.npy` files **verbatim** — `InteriorDomain` is whatever the
file contains, with no unit conversion applied:

> Why this is set up the way it is: [HAT_hindcast_methods.md](docs/HAT_hindcast_methods.md#24-units-check-against-barrier3ds-input-contract)

Done by `setup_topography` above: the contract it read and every check are in its report.

## 3. Island orientation — set `START_YEAR`

`START_YEAR` is the one flip in this notebook. It selects a period from
`HATTERAS_PERIODS` (in `hatteras_site_config.py`), and everything in Section 4
follows from it: run length, RSLR rate, storm series, background-erosion
preset, plus the road-setback and nourishment settings used later.

> Why this is set up the way it is: [HAT_hindcast_methods.md](docs/HAT_hindcast_methods.md#3-island-orientation-set-startyear)

In [ ]:
# The period, the scenario switches, where the run is filed, and the run-name preview, all from
# hat_run.yaml (RUN_CONFIG). Prints the settings and their sources, the preflight and the scenario report.
setup_scenario(r)

### 3.1 Island orientation QC plot

Both periods are drawn so the active one is visible *in context* — the 1984
and 2004 lines should have the same overall shape (it is the same island) but
sit at different cross-shore positions. The lower panel is the difference
between the two files. It is **not** shoreline change:
`step2_raw_offsets_to_island_offset.py:125` zeroes each year on its own most-seaward domain,
and that domain moved between the two surveys, so the difference carries a
constant offset (56.4 m for 1984-2004, enough to flip the sign of the mean).
Read it for the alongshore *pattern* only. Section 9.4 rebuilds the real change
from the raw transect files, which share a fixed datum.

Buffer domains are excluded here; their offsets are extrapolated ramps rather
than measured, so they would dominate the y-range. See the poster figure in
`scripts/figure_making/island/` for the buffered plan view.

In [ ]:
# Each padded domain's cross-shore starting position, in metres: shoreline_offset on the Cascade() call
setup_island_offset(r)

# Setup QC figure: every period's offsets, the active one in context. Nothing downstream reads it.
# Built as the run builds them (ring re-closed for this geometry), in decametres for the plot; a retired
# period whose file is gone is left out
_offsets_by_year = {
    year: build_island_offset(r.HATTERAS_DATA_BASE / period["island_offset_file"],
                              HATTERAS_DOMAINS) / DAM_TO_M
    for year, period in HATTERAS_PERIODS.items()
    if (r.HATTERAS_DATA_BASE / period["island_offset_file"]).is_file()
}
_ = setup_qc.plot_island_orientation(_offsets_by_year, r.START_YEAR, HATTERAS_DOMAINS)
plt.show()

### 3.2 Initialization plan view

Section 2's topography and Section 3's offsets combined: every domain's
elevation array shifted cross-shore by its island offset, so the initial
island reads as a map. This is the last check before the forcings — if the
orientation year is wrong, or a domain's topography is misaligned, it shows
up here as a kink in the island rather than as a number.

Both variants are drawn. The buffered one shows the 15 interpolated domains
on each end; they repeat one sample profile stepped along the extrapolated
offset ramp, which is what makes the ramp itself visible.

Compositing — cross-shore padding, unit conversion, canvas assembly — comes
from `cascade_pipeline.plotting.init_planview`, which
`scripts/figure_making/island/initialization_figures.py`
also uses, so the two cannot disagree about the surface. Only the styling
differs: that script is tuned for a printed page, this for a notebook cell.

In [ ]:
PLAN_VIEW = init_planview.PlanViewConfig()  # extractor defaults: 200 rows, -3.0 m water

# Setup QC figure. Nothing downstream reads it -- comment it out to skip.
# PLAN_VIEW itself is read again by 5.2, so it stays defined either way.
# The plan view takes decametres: Barrier3D's frame, brie.x_s / 10. Cascade
# adds island_offset to brie.x_s in metres, so that frame is island_offset / 10.
_ = setup_qc.plot_initialization_planview(
    r.ELEVATION_FILE_PATHS, r.island_offset / DAM_TO_M, HATTERAS_DOMAINS, r.START_YEAR,
    config=PLAN_VIEW)
plt.show()

## 4. Period forcings — RSLR, storms, source/sink

Everything in this section is resolved by the `START_YEAR` set in Section 3.
Re-run Section 3 with the other year and re-run these cells: every number and
plot below should change together. That is the check — a forcing that does
*not* change is one that is not actually period-dependent.

In [ ]:
# 4.1-4.3: the sea-level rise rate, the storm series and the source/sink rates, each with its report
setup_forcings(r)

### 4.1 Relative sea level rise

A single rate in m/yr, applied every model year. Both periods are plotted from
their own start year so the fork is visible: the 2004 period rises faster
(0.006 vs 0.004 m/yr) over the same 20-year run length.

In [ ]:
# Setup QC figure. Nothing downstream reads it -- comment it out to skip.
_ = setup_qc.plot_sea_level_rise(HATTERAS_PERIODS, r.START_YEAR)
plt.show()

### 4.2 Storm series

Barrier3D's storm file is one row per storm with columns
`time, Rhigh, Rlow, period, duration`. `time` is the model time step the storm
lands on (1-based), `Rhigh`/`Rlow` are runup elevations in **decameters**, and
`duration` is in hours.

Two things to check: the time axis should span the run (roughly 1 to
`RUN_YEARS`, with no storms past the end), and `Rhigh` should look like metres
of runup once converted — a file left in metres would show up here as values
ten times too large.

In [ ]:
def plot_storm_series(series, start_year, run_years):
    """Plots storm runup and storm counts against the model time axis.

    Args:
        series: DataFrame from load_storm_series.
        start_year: Calendar year of model time step 0.
        run_years: Number of model years in the run.

    Returns:
        The matplotlib Figure.
    """
    fig, (ax_runup, ax_count) = plt.subplots(
        2, 1, figsize=(13, 7), sharex=True,
        gridspec_kw={"height_ratios": [2, 1]})

    ax_runup.scatter(series["time"], series["Rhigh_m"], s=18, alpha=0.7,
                     label="Rhigh")
    ax_runup.scatter(series["time"], series["Rlow_m"], s=18, alpha=0.7,
                     label="Rlow")
    ax_runup.set_ylabel("Runup elevation (m)")
    ax_runup.set_title(f"Storm series, {start_year}-{start_year + run_years}")
    ax_runup.legend()

    counts = series["time"].value_counts().sort_index()
    ax_count.bar(counts.index, counts.values, width=0.7)
    ax_count.set_ylabel("Storms per year")
    ax_count.set_xlabel("Model time step (year 1 = "
                        f"{start_year + 1})")

    for ax in (ax_runup, ax_count):
        ax.axvline(run_years + 0.5, color="#c0392b", ls="--", lw=1.2,
                   label="_run end")
        ax.grid(alpha=0.3)

    fig.tight_layout()
    return fig


_ = plot_storm_series(r.STORM_SERIES, r.START_YEAR, r.RUN_YEARS)
plt.show()

### 4.3 Source/sink (background erosion)

Background erosion is CASCADE's stand-in for shoreline change driven by
alongshore transport gradients that the model does not resolve. It is a per
domain rate in m/yr, passed to Barrier3D as `Rat`. Sign convention, from
`cascade/brie_coupler.py`: **(-) = erosion, (+) = accretion**.

> Why this is set up the way it is: [HAT_hindcast_methods.md](docs/HAT_hindcast_methods.md#43-sourcesink-background-erosion)

In [ ]:
def plot_background_erosion(presets, start_year, active_preset, geometry):
    """Plots every background-erosion preset for one period.

    Args:
        presets: Mapping of preset name to {start_year: {gis_id: rate}}.
        start_year: The period to plot.
        active_preset: Name of the preset currently selected.
        geometry: DomainGeometry used to slice out the real domains.

    Returns:
        The matplotlib Figure.
    """
    real = slice(geometry.start_real_index, geometry.end_real_index)
    gis_ids = np.arange(geometry.first_gis_id, geometry.last_gis_id + 1)

    fig, (ax_full, ax_zoom) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

    interior = slice(1, -1)   # drop the locked end domains for the zoom range
    interior_rates = []

    for name, by_period in presets.items():
        rates = np.array(build_background_erosion(
            by_period[start_year], geometry))[real]
        interior_rates.append(rates[interior])
        is_active = name == active_preset
        style = dict(lw=2.0 if is_active else 1.2,
                     alpha=1.0 if is_active else 0.5,
                     marker="o" if is_active else None, ms=3,
                     label=f"{name}" + ("  (active)" if is_active else ""))
        ax_full.plot(gis_ids, rates, **style)
        ax_zoom.plot(gis_ids, rates, **style)

    # GIS 1 and 90 are locked to large solved values that would otherwise
    # flatten the interior detail, so the lower panel excludes them.
    margin = 0.5
    ax_zoom.set_ylim(min(r.min() for r in interior_rates) - margin,
                     max(r.max() for r in interior_rates) + margin)

    ax_full.set_title(f"Source/sink rates, {start_year} period "
                      "     (-) erosion  /  (+) accretion")
    ax_full.set_ylabel("Background erosion (m/yr)")
    ax_full.legend()

    ax_zoom.set_title(f"Zoomed to GIS {geometry.first_gis_id + 1}-"
                      f"{geometry.last_gis_id - 1} (locked end domains "
                      "off-scale)")
    ax_zoom.set_ylabel("Background erosion (m/yr)")
    ax_zoom.set_xlabel("GIS domain (S -> N, Cape Point to Pea Island)")

    for ax in (ax_full, ax_zoom):
        ax.axhline(0, color="k", lw=0.8)
        ax.grid(alpha=0.3)
    fig.tight_layout()
    return fig


# Only the presets this period has (calibBE is fitted for 1984 and 2004 only; domainBE for 1996 and 2009)
_ = plot_background_erosion(
    {name: by_period for name, by_period in HATTERAS_BE_PRESETS.items() if r.START_YEAR in by_period},
    r.START_YEAR, r.SOURCE_SINK_PRESET, HATTERAS_DOMAINS)
plt.show()

## 5. `roadway_manager` — setbacks, per-domain elevation, historical events

NC-12's forcing is three things: **where** the road sits (setback, by period),
**how high** it is (elevation, period-independent), and **which domains are
managed** at all.

> Why this is set up the way it is: [HAT_hindcast_methods.md](docs/HAT_hindcast_methods.md#5-roadwaymanager-setbacks-per-domain-elevation-historical-events)

In [ ]:
# NC-12's setbacks (by period), elevations and managed domains, and the year-one audit of 5.1. Loaded
# and audited even with roadway management off; it then never reaches a RoadwayManager.
setup_roadway(r)

### 5.1 Pre-flight audit — which roads will not survive year one

`bulldoze` tests the two rows *flanking* the road — never the road's own cells
— and drowns it when either flank is more than 20 % water.

A drowned road is not a warning. `roadway_manager` sets `_drown_break` and
returns immediately on every later year, so the domain gets no overwash
removal, no dune rebuilding and no relocation for the rest of the run. It
becomes an **unmanaged barrier wearing a road label**, which is why these
domains are named here rather than discovered in the output: any
managed-versus-unmanaged comparison has to exclude them explicitly.

The audit is the last part of `setup_roadway`'s report above.

### 5.2 The roadway on the initialization surface

The same plan view as §3.2, with NC-12 drawn where CASCADE puts it. Each
domain gets a horizontal bar at `offset + setback`, because that is literally
what the model does — `road_start = int(setback / cell)` is one row index
applied to every alongshore profile in the domain.

The orange bar is the position after the prescribed relocations, evaluated at
t=0. That is the **upper bound**: CASCADE decrements the setback by dune
migration between now and the event year, so the real relocated position sits
seaward of the bar drawn here. Blue markers flag the domains whose road drowns
in year one.


In [ ]:
# Display only - nothing in this cell feeds the run.
road_setbacks_relocated = r.road_setbacks_full.copy()
for _event in HATTERAS_ROAD_EVENTS:
    if isinstance(_event, roadway.RelocationEvent) and _event.enabled:
        road_setbacks_relocated = roadway.relocated_setbacks(
            road_setbacks_relocated, _event, HATTERAS_DOMAINS)

road_offset_cells = np.round(
    r.island_offset * DAM_TO_M / PLAN_VIEW.cell_size_m).astype(int)

_fig = plot_roadway_planview(
    r.ELEVATION_FILE_PATHS,
    road_offset_cells,
    r.road_setbacks_full,
    HATTERAS_DOMAINS,
    title=(f"NC-12 on the CASCADE initialization  |  setbacks {r.START_YEAR}, "
           f"topography {r.TOPO_PRODUCT}/{r.TOPO_DUNE_VERSION}, dune offsets applied"),
    relocated_m=road_setbacks_relocated,
    drowning_gis=r.audit_summary["drowning"],
    config=PLAN_VIEW,
    xlabel="GIS domain (S -> N, Cape Point to Pea Island)",
)
plt.show()

## 6. `beach_dune_manager` — nourishment schedule + overwash filter

Two different things arrive bundled in one CASCADE module, and the distinction
matters more than the name suggests.

> Why this is set up the way it is: [HAT_hindcast_methods.md](docs/HAT_hindcast_methods.md#6-beachdunemanager-nourishment-schedule-overwash-filter)

### 6.1 Build the schedule, filter, and management footprints
Everything CASCADE is handed for this period: the period-filtered project
schedule, the overwash filter, and the two per-domain module footprints.
`NOURISHMENT_VOLUME_INIT` is deliberately zero -- see the comment in `setup_beach_dune`.

In [ ]:
# The period's fill schedule (and the one actually applied), the overwash filter, both managers'
# footprints, and the schedule audit, with the beach/dune report of 6.3
setup_beach_dune(r)

### 6.2 Plot helpers
`contiguous_gis_runs` is generic domain geometry; `plot_beach_dune_management`
draws the two panels below. Both are candidates to move into
`cascade_pipeline` -- nothing in this cell produces output.


In [ ]:
def contiguous_gis_runs(mask, geometry):
    """Contiguous GIS spans where a padded boolean mask is True.

    Args:
        mask: Sequence of booleans indexed by padded position.
        geometry: DomainGeometry describing the padded array.

    Returns:
        A list of inclusive (first_gis, last_gis) tuples.
    """
    runs = []
    start = None
    for pad in range(geometry.start_real_index, geometry.end_real_index):
        if mask[pad] and start is None:
            start = pad
        elif not mask[pad] and start is not None:
            runs.append((int(geometry.pad_to_gis(start)),
                         int(geometry.pad_to_gis(pad - 1))))
            start = None
    if start is not None:
        runs.append((int(geometry.pad_to_gis(start)),
                     int(geometry.pad_to_gis(geometry.end_real_index - 1))))
    return runs


def plot_beach_dune_management(overwash_filter, beach_dune_on, roadway_on,
                               schedule, double_managed_gis, geometry):
    """Plots the overwash filter and both management footprints by GIS domain.

    Two panels on a shared domain axis. The top panel is the magnitude the
    module actually applies; the bottom is which modules run where, so the
    overlap that neither footprint was drawn to avoid is visible rather than
    inferred.

    Args:
        overwash_filter: Per-domain filter percentages.
        beach_dune_on: Per-domain booleans for beach_nourishment_module.
        roadway_on: Per-domain booleans for roadway_management_module.
        schedule: The NourishmentSchedule for this period.
        double_managed_gis: GIS domains where both modules run.
        geometry: DomainGeometry describing the padded array.

    Returns:
        The matplotlib Figure.
    """
    c_filter = "#08519C"    # deep blue, matches the notebook's obs family
    c_bdm = "#08519C"
    c_road = "#6BAED6"      # lighter step of the same hue
    c_nourish = "#FF8C00"   # warm accent, as elsewhere for model-side forcing
    c_double = "#B71C1C"

    real = slice(geometry.start_real_index, geometry.end_real_index)
    gis_ids = np.arange(geometry.first_gis_id, geometry.last_gis_id + 1)
    filter_real = np.asarray(overwash_filter, dtype=float)[real]

    nourished_pads = {geometry.gis_to_pad(g) for g in schedule.nourished_gis}
    nourished_mask = [pad in nourished_pads
                      for pad in range(geometry.total_domains)]

    fig, (ax_filter, ax_map) = plt.subplots(
        2, 1, figsize=(13, 6), sharex=True,
        gridspec_kw=dict(height_ratios=[1.4, 1]))

    # --- top: how much overwash is filtered, and where -----------------------
    # The shading is the point of the panel: nourished ground and filtered
    # ground only partly coincide, and the domains that are shaded but have no
    # bar are the ones running the module for the fill alone.
    for first_gis, last_gis in contiguous_gis_runs(nourished_mask, geometry):
        ax_filter.axvspan(first_gis - 0.5, last_gis + 0.5, color=c_nourish,
                          alpha=0.15, zorder=0, label="_nolegend_")
    ax_filter.bar(gis_ids, filter_real, width=0.9, color=c_filter,
                  label="overwash filtered (developed ground)")
    if schedule.nourished_gis:
        ax_filter.axvspan(np.nan, np.nan, color=c_nourish, alpha=0.15,
                          label="nourished at least once")
    ax_filter.set_ylabel("overwash_filter (%)")
    ax_filter.set_ylim(0, max(50, filter_real.max() * 1.25))
    ax_filter.set_title(
        f"beach_dune_manager forcing, {schedule.start_year}-"
        f"{schedule.end_year}     "
        f"filter {HATTERAS_BEACH_DUNE.community_overwash_filter_pct:.0f}% + "
        f"{HATTERAS_BEACH_DUNE.overwash_to_dune_pct:.0f}% to dune")
    ax_filter.legend(loc="upper left", frameon=False)
    ax_filter.grid(alpha=0.3, axis="y")

    # --- bottom: which module runs where -------------------------------------
    rows = [
        ("roadway_manager", roadway_on, c_road, 0),
        ("beach_dune_manager", beach_dune_on, c_bdm, 1),
    ]
    for label, mask, color, y in rows:
        for first_gis, last_gis in contiguous_gis_runs(mask, geometry):
            ax_map.barh(y, last_gis - first_gis + 1, left=first_gis - 0.5,
                        height=0.55, color=color, align="center")
        ax_map.text(geometry.first_gis_id - 1.5, y, label, ha="right",
                    va="center", fontsize=9)

    # Overlap: outlined rather than filled, so the underlying rows stay legible.
    for first_gis, last_gis in contiguous_gis_runs(
            [beach_dune_on[p] and roadway_on[p]
             for p in range(geometry.total_domains)], geometry):
        ax_map.add_patch(plt.Rectangle(
            (first_gis - 0.5, -0.45), last_gis - first_gis + 1, 1.9,
            fill=False, edgecolor=c_double, lw=1.8, hatch="///", zorder=5))

    # Nourishment events, labelled per project rather than per year -- two
    # projects share 2022, and a single averaged label would sit between them,
    # over neither.
    for row in schedule.events():
        ax_map.plot(row["gis"], 2, marker="v", ms=8, color=c_nourish, zorder=6)
    for project in schedule.projects:
        ax_map.text(float(np.mean(project.gis_domains)), 2.4,
                    str(project.year), ha="center", va="bottom", fontsize=9,
                    color=c_nourish)
    ax_map.text(geometry.first_gis_id - 1.5, 2, "nourishment", ha="right",
                va="center", fontsize=9, color=c_nourish)

    if double_managed_gis:
        ax_map.legend(
            handles=[plt.Rectangle((0, 0), 1, 1, fill=False,
                                   edgecolor=c_double, lw=1.8, hatch="///")],
            labels=[f"both modules ({len(double_managed_gis)} domains)"],
            loc="upper left", frameon=False, fontsize=9)

    ax_map.set_ylim(-0.7, 3.4)
    ax_map.set_yticks([])
    ax_map.set_xlim(geometry.first_gis_id - 6, geometry.last_gis_id + 1)
    ax_map.set_xticks(np.arange(geometry.first_gis_id,
                                geometry.last_gis_id + 1, 5))
    ax_map.set_xlabel("GIS domain (S -> N, Cape Point to Pea Island)")
    ax_map.grid(alpha=0.3, axis="x")
    for spine in ("top", "right", "left"):
        ax_map.spines[spine].set_visible(False)

    fig.tight_layout()
    return fig

### 6.3 Audit and QC plot
The schedule as it will be applied, the pre-flight audit, and the
double-management warning. The plot shows the overwash filter over the
footprints, so the overlap neither footprint was drawn to avoid is visible
rather than inferred.

The schedule and audit are `setup_beach_dune`'s report in 6.1.

In [ ]:
_ = plot_beach_dune_management(
    r.OVERWASH_FILTER, r.BEACH_DUNE_MANAGEMENT_ON, r.ROADWAY_MANAGEMENT_ON,
    r.BN_SCHEDULE_APPLIED, r.DOUBLE_MANAGED_GIS, HATTERAS_DOMAINS)
plt.show()

## 7. `hard_structures` / groin -- Buxton groin field

`cascade/groin.py`'s `GroinManager` is a CASCADE module like `BeachDuneManager`: `Cascade(groin_module=True,
groin_updrift_pad=..., groin_downdrift_pad=..., groin_strength_history=..., ...)` builds it, and
`Cascade.update()` calls `GroinManager.update()` once per model year, immediately before the
alongshore-transport solve. Each active year it moves a share of BRIE's wave-driven drift
arriving at the GIS 5|6 face from the downdrift domain to the updrift one, and cancels BRIE's diffusive
leak of that sand on the groin's own bends (measured against the groin's tracked footprint, so a groin of
strength 0 is exactly the no-groin run). Two numbers set it: the strength, 0.25, and the first year it no
longer works, 2017 (the Buxton fill), both fitted on one continuous 1967-2025 run
(`hard-structures/groin/4-hindcast/3-wave-driven-1967-2025/2026-10-13-pregroin-planform/`). The dipole (M)
and blocking (b, f) groins this replaced were removed on 2026-10-14.

> Why this is set up the way it is: [HAT_hindcast_methods.md](docs/HAT_hindcast_methods.md#7-hardstructures-groin----buxton-groin-field)

### 7.1 Switches and structure
Every constant the groin needs, and nothing that acts on them. The
`GROIN_ENABLED` guard fails fast here if the Cascade in use has no
`groin_module` argument (no `GroinManager`).

In [ ]:
# 7.1-7.4: the guard that this Cascade has a GroinManager, the Buxton groin's domains and install year,
# its strength, switch-off year and spread from hat_run.yaml, the settings Cascade builds it from, and
# the groin report
setup_groin(r)

### 7.2 The GroinManager's settings
Drawn up unconditionally so 7.4's QC output renders either way. Cascade builds
its `GroinManager` from them in section 11 (`groin_module=GROIN_ENABLED`); after
the build it is `cascade.groin`, and `None` when the groin is off.

Drawn up by `setup_groin` above: `r.GROIN_UPDRIFT_PAD`, `r.GROIN_DOWNDRIFT_PAD`, `r.GROIN_STRENGTH_HISTORY`.

### 7.3 Diagnostic and plot helpers
`groin_trapping_schedule` reads the strength history the `GroinManager` will
be built with, before the Cascade exists.

In [ ]:


def plot_groin_setting(offset_m, strength_history, geometry, updrift_gis,
                       downdrift_gis, halfwidth, periods, active_year):
    """Plots the groin's alongshore setting and its strength in every period.

    Two panels: the initial shoreline the groin works on, either side of the
    structure, and the strength each period's run uses (the share of the
    arriving drift held back), so the switch-off year's place in the run is
    visible rather than inferred.

    Args:
        offset_m: Padded island-offset array, in metres.
        strength_history: The groin's (from_year, strength) pairs.
        geometry: DomainGeometry describing the padded array.
        updrift_gis: GIS domain that accretes.
        downdrift_gis: GIS domain that erodes.
        halfwidth: Domains either side of the groin to show.
        periods: Mapping of start year to a period dict carrying "end_year".
        active_year: The period currently selected.

    Returns:
        The matplotlib Figure.
    """
    from site_layer.hatteras_site_config import first_model_year, last_model_year
    c_shore, c_groin = "#08519C", "#B71C1C"
    boundary = (downdrift_gis + updrift_gis) / 2.0
    gis_lo = max(geometry.first_gis_id, downdrift_gis - halfwidth)
    gis_hi = min(geometry.last_gis_id, updrift_gis + halfwidth)
    gis_ids = np.arange(gis_lo, gis_hi + 1)
    shore = np.asarray(offset_m)[geometry.gis_to_pad(gis_ids)]

    fig, (ax_shore, ax_time) = plt.subplots(2, 1, figsize=(13, 7),
                                            gridspec_kw=dict(height_ratios=[1.4, 1]))
    ax_shore.plot(gis_ids, shore, color=c_shore, lw=2.0, marker="o", ms=4,
                  label=f"{active_year} island offset")
    ax_shore.axvline(boundary, color=c_groin, lw=1.6, ls=":", label=f"Buxton groin (GIS {boundary})")
    ax_shore.set_xlim(gis_lo - 0.6, gis_hi + 0.6)
    ax_shore.set_xticks(gis_ids)
    ax_shore.set_xlabel("GIS domain (S -> N)")
    ax_shore.set_ylabel("Island offset (m)")
    ax_shore.set_title(f"Buxton groin setting, GIS {gis_lo}-{gis_hi}: updrift D{updrift_gis} (north) / "
                       f"downdrift D{downdrift_gis} (south)")
    ax_shore.legend(loc="best", frameon=False, fontsize=9)
    ax_shore.grid(alpha=0.3, axis="y")

    for year in sorted(periods):
        years, s_eff = groin_trapping_schedule(strength_history, first_model_year(year),
                                               last_model_year(year) + 1)
        is_active = year == active_year
        ax_time.step(years, s_eff, where="post", lw=2.2 if is_active else 1.2, alpha=1.0 if is_active else 0.45,
                     label=f"{year}-{periods[year]['end_year']}" + ("  (active)" if is_active else ""))
    ax_time.set_ylim(0, max(s for _, s in strength_history) * 1.25 or 1)
    ax_time.set_ylabel("Strength (share of drift held)")
    ax_time.set_xlabel("Model year")
    ax_time.set_title("The groin's strength in each period")
    ax_time.legend(loc="best", frameon=False, fontsize=9)
    ax_time.grid(alpha=0.3)
    fig.tight_layout()
    return fig

### 7.4 Report -- configuration and double-count audit
The configured groin, its strength over the run, and the overlap with the
calibrated source/sink rates, reported, not corrected. The trapped volume is
emergent and is written per year to the groin diagnostics CSV.

The report is the last part of `setup_groin`'s output in 7.1.

In [ ]:
GROIN_PLOT_HALFWIDTH = 5   # domains either side of the groin in the QC plot

_ = plot_groin_setting(
    r.island_offset, r.GROIN_STRENGTH_HISTORY, HATTERAS_DOMAINS,
    r.GROIN_UPDRIFT_GIS, r.GROIN_DOWNDRIFT_GIS, GROIN_PLOT_HALFWIDTH,
    HATTERAS_PERIODS, r.START_YEAR)
plt.show()

### 7.5 Scenario summary -- every switch, and the run name derived from them

This notebook runs one scenario at a time, and the switches that decide which
one are spread across sections 3, 5, 6 and 7. Nothing below changes the model.
It exists because `RUN_NAME_SUFFIX` used to be typed by hand: run with the
groin off, forget to retype the label, and the output lands in a directory
named for a different experiment. The suffix is now derived from the switches
themselves, so the directory name cannot disagree with what was simulated.

One switch deliberately contributes no token: background erosion. It is
implied by the source/sink preset and checked against it in 4.3, so emitting
both produced names like `..._base_noBE_...` that said the same thing twice
without saying which zero it was.

Output directories from earlier runs (`HAT_1984_2004_full_calibrated`) will not
match the derived names, and neither will runs made before the presets were
renamed. That is the trade: the old names are shorter, the new ones are true.

In [ ]:
# Every switch that changes what is simulated, the run name derived from them, and the check that it
# matches section 3's preview
setup_run_name(r)

## 8. CoastSat target rates -- LOWESS windows

This section builds the observational target. The `calibBE` source/sink
preset in section 4.3 was fit against the curve produced here, and section 12
draws the model against it -- so what this section decides is what the model is
judged by. It emits `cs_series` for the section 12 figures and
`COASTSAT_TARGET`, a per-domain table of the target rate itself.

> Why this is set up the way it is: [HAT_hindcast_methods.md](docs/HAT_hindcast_methods.md#8-coastsat-target-rates----lowess-windows)

### 8.1 Load both periods' CoastSat series
Observational data, not simulation forcing, so both periods always load and
the active one is selected from them. A `START_YEAR` with no matching dataset
fails here rather than silently plotting the wrong period.


In [ ]:
# 8.1-8.4: every period's CoastSat series (the active one selected), the 7-domain LOWESS target as a
# table, and the CoastSat report
setup_coastsat_target(r)

### 8.2 The target, as a table
`build_target_table` resolves each domain to a LOWESS window or a raw rate;
`groin_differential` extracts the updrift-minus-downdrift signal that is the
observational check on the dipole section 7 imposes.

Built by `setup_coastsat_target` above: `r.COASTSAT_TARGET`, and `r.COASTSAT_TARGET_BASE` for the interior score.

### 8.3 Plot helper
`plot_coastsat_target` draws the target and its provenance. Defined
separately so it can be edited and re-run without rebuilding the table.


In [ ]:
# =============================================================================
# Plot
# =============================================================================
def plot_coastsat_target(cs, target, lowess_config, geometry, annotations,
                         config, window, updrift_gis, downdrift_gis,
                         zoom_to_gis=15):
    """Plots the active period's transect scatter and its smoothed target.

    Two panels. The top is the whole island with the geographic annotation
    layer, so the target can be read against the villages and shoal zones that
    explain its structure. The bottom zooms on the unsmoothed southern zone,
    where the target is per-domain means rather than a curve and where the
    groin sits.

    Colors come from RateComparisonConfig so this figure and the section 12
    model comparison stay in the same visual language.

    Args:
        cs: The active entry from build_coastsat_series.
        target: Output of build_target_table.
        lowess_config: LowessConfig used.
        geometry: DomainGeometry.
        annotations: AnnotationConfig for the geographic layer.
        config: RateComparisonConfig supplying colors and marker sizes.
        window: Reference window width, in domains.
        updrift_gis, downdrift_gis: The groin's flanking domains.
        zoom_to_gis: Last GIS domain shown in the lower panel.

    Returns:
        The matplotlib Figure.
    """
    skip = lowess_config.skip_southern_domains
    # Matches rate_comparison's GIS-domain x-axis exactly, so the scatter in
    # this figure lands where it lands in the section 12 comparison.
    scatter_x = (np.asarray(cs["transect_along_coast"])
                 / geometry.domain_spacing_m + geometry.first_gis_id)
    boundary = (updrift_gis + downdrift_gis) / 2.0

    fig, (ax_full, ax_zoom) = plt.subplots(
        2, 1, figsize=(14, 9),
        gridspec_kw=dict(height_ratios=[1.7, 1]))

    # --- top: the whole island ----------------------------------------------
    ax_full.scatter(scatter_x, cs["transect_rates"], color=config.raw_color,
                    s=config.raw_scatter_size, alpha=config.raw_scatter_alpha,
                    zorder=1, linewidths=0, label="transect LRR")
    for win in cs["windows"]:
        color = config.window_colors.get(win["window"],
                                         config.window_color_default)
        mask = win["gis_x"] > skip
        is_target = win["window"] == window
        ax_full.plot(win["gis_x"][mask], win["smoothed"][mask], color=color,
                     lw=2.2 if is_target else 1.6, zorder=4 if is_target else 3,
                     label=f"LOWESS {win['window']}-dom"
                           + ("  (target)" if is_target else ""))
    ax_full.axvspan(geometry.first_gis_id - 0.5, skip + 0.5, color="#AAAAAA",
                    alpha=0.16, zorder=0,
                    label=f"D1-{skip}: no LOWESS (raw means)")
    add_geographic_annotations(ax_full, annotations)
    ax_full.axhline(0, color="k", lw=0.8)
    ax_full.set_ylabel("Shoreline change rate (m/yr)")
    ax_full.set_title(f"{cs['label']}     target = LOWESS {window}-domain "
                      f"({window * geometry.domain_spacing_m / 1000:.1f} km) "
                      f"north of D{skip}, raw per-domain mean south of it")
    ax_full.set_xlim(geometry.first_gis_id - 1, geometry.last_gis_id + 1)
    ax_full.legend(loc="upper left", frameon=True, fontsize=8, ncol=2)
    ax_full.grid(alpha=0.3)

    # --- bottom: the unsmoothed zone, where the groin is ---------------------
    south = target[target["gis_domain"] <= zoom_to_gis]
    in_zoom = scatter_x <= zoom_to_gis + 1
    ax_zoom.scatter(scatter_x[in_zoom],
                    np.asarray(cs["transect_rates"])[in_zoom],
                    color=config.raw_color, s=config.raw_scatter_size * 3,
                    alpha=0.75, zorder=2, linewidths=0, label="transect LRR")
    raw = south[south["source"].str.startswith("raw")]
    smoothed_rows = south[~south["source"].str.startswith("raw")]
    ax_zoom.step(raw["gis_domain"], raw["target_lrr_m_yr"], where="mid",
                 color="#08519C", lw=2.0, zorder=4,
                 label=f"target: raw per-domain mean (D1-{skip})")
    if len(smoothed_rows):
        ax_zoom.plot(smoothed_rows["gis_domain"], smoothed_rows["target_lrr_m_yr"],
                     color="#08519C", lw=2.0, ls="--", zorder=4,
                     label=f"target: LOWESS {window}-dom (D{skip + 1}+)")
    for gis, color in ((updrift_gis, "#FF8C00"), (downdrift_gis, "#6BAED6")):
        ax_zoom.axvspan(gis - 0.5, gis + 0.5, color=color, alpha=0.20, zorder=0)
    ax_zoom.axvline(boundary, color="#B71C1C", lw=1.6, ls=":", zorder=5,
                    label=f"Buxton groin (GIS {boundary})")
    ax_zoom.axhline(0, color="k", lw=0.8)
    ax_zoom.set_xlim(geometry.first_gis_id - 0.6, zoom_to_gis + 0.6)
    ax_zoom.set_xticks(range(geometry.first_gis_id, zoom_to_gis + 1))
    ax_zoom.set_ylabel("Shoreline change rate (m/yr)")
    ax_zoom.set_xlabel("GIS domain (S -> N, Cape Point to Pea Island)")
    ax_zoom.set_title(f"Unsmoothed southern zone, D{geometry.first_gis_id}-"
                      f"{zoom_to_gis}: the groin's target is a mean over a "
                      f"handful of transects")
    ax_zoom.legend(loc="best", frameon=True, fontsize=8)
    ax_zoom.grid(alpha=0.3)

    fig.tight_layout()
    return fig

### 8.4 Report and QC plot
How many domains came from a LOWESS window versus a raw rate, the groin
differential, and the figure.

The report is `setup_coastsat_target`'s output in 8.1.

In [ ]:
_ = plot_coastsat_target(
    r.CS_ACTIVE, r.COASTSAT_TARGET, r.LOWESS_CONFIG, HATTERAS_DOMAINS,
    HATTERAS_ANNOTATIONS, DEFAULT_RATE_COMPARISON, r.TARGET_WINDOW,
    r.GROIN_UPDRIFT_GIS, r.GROIN_DOWNDRIFT_GIS)
plt.show()

## 9. Rate conventions, and the groin's paired baseline

The sign convention (`FLIP_SIGN_MODEL`: up is seaward), the rate estimator the run is scored with
(`RATE_ESTIMATOR = "lrr"`, the same OLS slope the CoastSat target is), the LRR fit-quality threshold, and,
for a groin run, the no-groin twin its extent is measured against in 12.3.

The figure configuration that used to sit here (keyword bundles, animation jobs, the validation target)
moved with the figures into `figure_making/model_output/run_figures.py` on 2026-10-23.

> Why this is set up the way it is: [HAT_hindcast_methods.md](docs/HAT_hindcast_methods.md#9-figure-configuration)

## 10. `build_cascade` and `run_cascade_simulation`

Two functions. `build_cascade` constructs the model and attaches the groin;
`run_cascade_simulation` steps it through the period, applying historical
management, and writes the run's artifacts. Both take every input explicitly
rather than closing over notebook state, so the sweep scripts can call them
too.

> Why this is set up the way it is: [HAT_hindcast_methods.md](docs/HAT_hindcast_methods.md#10-define-runcascadesimulation)

In [ ]:
setup_conventions(r)

## 11. Initialize Cascade -- single config, no sweep

Section 10 defined the procedure; this section supplies the values sections
2-9 do not already produce and builds the model. Section 12 steps it.

> Why this is set up the way it is: [HAT_hindcast_methods.md](docs/HAT_hindcast_methods.md#11-initialize-cascade----single-config-no-sweep)

### 11.1 Parameters sections 2-9 do not produce
Dune growth bounds, rebuild thresholds, wave climate, datums, and run
identity. One configuration only -- a sweep over `Hs` is a separate script.


In [ ]:
# Dune growth bounds (per period), the dune rebuild thresholds, the road, the wave climate, the datums
# and the sandbag switch
setup_parameters(r)

# Run identity, from section 7.5's derived name. OVERWRITE (hat_run.yaml): false stops on an existing
# result; true EMPTIES the directory and reuses it -- set it back to false before running the matrix.
OVERWRITE = RUN_CONFIG.overwrite
r.RUN_NAME = r.RUN_NAME_BASE
r.RUN_DIR = str(r.OUTPUT_BASE_DIR / r.RUN_NAME)
# Read before the guard, which empties the directory when OVERWRITE is set
_replacing = run_dir_contents(r.RUN_DIR)
guard_run_dir(r.RUN_DIR, overwrite=OVERWRITE)
print(f"\nRUN_DIR               {r.RUN_DIR}"
      + ("   (OVERWRITE=True)" if OVERWRITE else ""))
if _replacing:
    print(f"  replaced            {len(_replacing)} file(s) from the previous run")

### 11.2 Build
The single `build_cascade` call. Every argument comes from a section above;
none is defined here.


In [ ]:
import shutil

# Each run gets its own parameter file (CASCADE rewrites it while it builds); the template stays read-only
r.RUN_PARAMETER_FILE = Path(r.RUN_DIR) / f"{r.RUN_NAME}-parameters.yaml"
shutil.copyfile(r.HATTERAS_DATA_BASE / r.PARAMETER_FILE, r.RUN_PARAMETER_FILE)

r.cascade = build_cascade(
    run_years=r.RUN_YEARS,
    name=r.RUN_NAME,
    storm_file=str(r.STORM_FILE),
    alongshore_section_count=HATTERAS_DOMAINS.total_domains,
    num_cores=r.NUM_CORES,
    rmin=r.RMIN, rmax=r.RMAX,
    elevation_file=r.ELEVATION_FILE_PATHS,
    dune_file=r.DUNE_FILE_PATHS,
    dune_design_elevation=r.DUNE_DESIGN_ELEVATION,
    dune_minimum_elevation=r.DUNE_MINIMUM_ELEVATION,
    road_ele=r.ROAD_ELEVATION,
    road_width=r.ROAD_WIDTH,
    road_setback=r.road_setbacks_full,
    overwash_filter=r.OVERWASH_FILTER,
    overwash_to_dune=r.OVERWASH_TO_DUNE,
    nourishment_volume=r.NOURISHMENT_VOLUME_INIT,
    background_erosion=r.BACKGROUND_EROSION_RATES,
    roadway_management_on=r.ROADWAY_MANAGEMENT_ON,
    beach_dune_manager_on=r.BEACH_DUNE_MANAGEMENT_ON,
    sea_level_rise_rate=r.SEA_LEVEL_RISE_RATE,
    sea_level_constant=r.SEA_LEVEL_CONSTANT,
    sandbag_management_on=r.SANDBAG_MANAGEMENT_ON,
    sandbag_elevation=r.SANDBAG_ELEVATION,
    enable_shoreline_offset=True,
    shoreline_offset=r.island_offset,
    wave_height=r.Hs,
    wave_period=r.FIXED_WAVE_PERIOD,
    wave_asymmetry=r.FIXED_WAVE_ASYMMETRY,
    wave_angle_high_fraction=r.FIXED_WAVE_ANGLE_HIGH_FRACTION,
    berm_elevation=r.BERM_ELEVATION,
    MHW=r.MHW_ELEVATION,
    beta=r.BEACH_SLOPE,
    data_base=r.HATTERAS_DATA_BASE,
    parameter_file=str(r.RUN_PARAMETER_FILE),
    relocation_setback_m=r.RELOCATION_SETBACK_M,
    road_dune_rebuild_trigger_m=r.ROAD_DUNE_REBUILD_TRIGGER_M,
    village_dune_cap_navd88_m=r.VILLAGE_DUNE_CAP_NAVD88_M,
    groin_module=r.GROIN_ENABLED,
    groin_updrift_pad=r.GROIN_UPDRIFT_PAD,
    groin_downdrift_pad=r.GROIN_DOWNDRIFT_PAD,
    groin_strength_history=r.GROIN_STRENGTH_HISTORY,
    groin_start_year=r.FIRST_MODEL_YEAR,
    groin_spread_weights=r.GROIN_SPREAD_WEIGHTS,
)

### 11.3 Pre-run diagnostics -- and the groin prediction
Reads the constructed model rather than the yaml, which resolves the active
profile height section 7 had to leave open. The predicted fillet amplitude and
extent are written down **before** the run; section 12 checks the emergent
extent against them. Amplitude was tuned, extent was not.


In [ ]:
# The active profile height read off the built model, and the groin's r_ipl, written down before the run
pre_run_diagnostics(r)

## 12. Run the loop, verify, then figures

Steps the model section 11 built, checks what it actually did, and only then
draws anything.

> Why this is set up the way it is: [HAT_hindcast_methods.md](docs/HAT_hindcast_methods.md#12-run-the-loop-verify-then-figures)

### 12.1 Run
The only expensive cell in the notebook. 12.2-12.5 read the finished model, so
a change to a check or a figure re-runs only those.

The guard is deliberate: Barrier3D seeds `x_s_TS` with one entry and appends
one per update, so stepping an already-stepped model would run past TMAX and
raise from deep inside Barrier3D. Re-run section 11 for a fresh model.


In [ ]:
from tqdm.auto import tqdm

# Barrier3D seeds x_s_TS with one entry and appends one per update, so a model
# that has already been stepped has more than one. Stepping it again would run
# past TMAX and raise from deep inside Barrier3D.
if len(r.cascade.barrier3d[0].x_s_TS) > 1:
    raise RuntimeError(
        f"this cascade has already been stepped "
        f"({len(r.cascade.barrier3d[0].x_s_TS)} states). Re-run section 11 to "
        f"build a fresh model before running section 12 again.")

_t0 = time.perf_counter()
with tqdm(total=r.RUN_YEARS, desc=f"{r.RUN_NAME}", unit="yr") as _bar:
    r.cascade = run_cascade_simulation(
        cascade=r.cascade,
        run_years=r.RUN_YEARS,
        name=r.RUN_NAME,
        run_dir=r.RUN_DIR,
        start_year=r.FIRST_MODEL_YEAR,
        geometry=HATTERAS_DOMAINS,
        alongshore_section_count=HATTERAS_DOMAINS.total_domains,
        historical_road_events=HATTERAS_ROAD_EVENTS,
        relocations_enabled=r.ENABLE_HISTORICAL_ROAD_RELOCATIONS,
        setback_check=HATTERAS_RELOCATION_CHECK_2004,
        nourishment_schedule=r.BN_SCHEDULE_APPLIED,
        progress=_bar,
        save_model_state=r.SAVE_MODEL_STATE,
    )
r.RUN_SECONDS = time.perf_counter() - _t0
print(f"\nruntime               {r.RUN_SECONDS / 60:.1f} min "
      f"({r.RUN_SECONDS / r.RUN_YEARS:.1f} s per model year)")

### 12.2 Verify
Checks what the model actually did before anything is drawn: run length
against the `states - 1` denominator that produced a 5% bias in earlier runs,
nourishment read from each manager's own `_nourishment_volume_TS`, the frozen
setbacks section 6 predicted, and which roads survived.


In [ ]:
# 12.2-12.3: run length, the shoreline matrix and both rate estimators, nourishment from the managers'
# own records, the frozen setbacks, which roads survived, and the groin's extent against its twin
verify_run(r)

### 12.3 Groin: the pre-registered extent check
The emergent alongshore extent against a paired no-groin baseline, compared to
the prediction made in 11.3. This is the part that could have failed --
amplitude was tuned, extent was not. Skipped, with a reason, when the groin is
off or no baseline run exists.

Part of `verify_run` in 12.2.

### 12.4 Write

The shoreline matrix, the overwash and road-relocation arrays the figures need, the change-rate CSV, the
road-management summary, the skill against the section 8 target, and the run metadata (its `index row`
section is what `run_registry.scan_runs` reads back).

> Why this is set up the way it is: [HAT_hindcast_methods.md](docs/HAT_hindcast_methods.md#124-write)

In [ ]:
write_run_outputs(r)

### 12.5 Figures

Drawn by `make_run_figures` from what 12.4 wrote into the run folder -- the same call `HAT_hindcast.py`
makes, so a figure here is the figure on disk. Shown inline because `SHOW_FIGURES` is true in a notebook.

In [ ]:
# The rate comparisons, the net change, overwash against the imagery, the end-year target misfit, and
# the animations, from the run folder
GIF_PATHS = make_run_figures(r.RUN_DIR, gifs=RUN_CONFIG.make_gifs, show=SHOW_FIGURES,
                             cs_series=r.cs_series)
print(f"\ndone                  {r.RUN_DIR}")

### 12.6 Animations, inline
`make_run_figures` writes the GIFs into the run folder's `animations/` and returns their paths;
this plays them here. Each embedded animation is written into the `.ipynb`
itself, so `GIF_INLINE_MAX_MB` caps how large a file gets inlined -- anything
over that is named, not embedded.

Re-runnable on its own: with no GIFs made this session it falls back to
whatever the run folder already holds, so a finished run can be reviewed without
stepping the model again.

In [ ]:
# =============================================================================
# Animations, inline
# =============================================================================
# A still PNG can be resampled to a display width before embedding. An
# animated GIF cannot, without re-encoding every frame -- so the file's own
# bytes are embedded and the width is set on the <img> tag instead.
GIF_DISPLAY_WIDTH = 900   # rendered width; source frames are ~1000-2000 px
GIF_INLINE_MAX_MB = 4.0   # per file -- embedded animations live in the .ipynb


def show_gif(path, width=GIF_DISPLAY_WIDTH, max_mb=GIF_INLINE_MAX_MB):
    """Plays one saved GIF inline, embedded as a data URI.

    Args:
        path: Path to the .gif.
        width: Display width in pixels.
        max_mb: Files larger than this are reported but not embedded, so one
            long high-stride animation cannot bloat the notebook.

    Returns:
        True if the animation was embedded, False if it was skipped.
    """
    path = Path(path)
    size_mb = path.stat().st_size / 1e6
    print(f"{path.name}  ({size_mb:.1f} MB)")
    if size_mb > max_mb:
        print(f"  over GIF_INLINE_MAX_MB={max_mb} -- not embedded; open {path}")
        return False
    data = base64.b64encode(path.read_bytes()).decode("ascii")
    display(HTML(f'<img src="data:image/gif;base64,{data}" width="{width}">'))
    return True


# Fall back to what is already on disk, so this cell alone re-displays a
# finished run without re-running section 12.1.
_gifs = [Path(p) for p in GIF_PATHS] or sorted(Path(r.RUN_DIR).rglob("*.gif"))
if not _gifs:
    print(f"no GIFs in {r.RUN_DIR} -- check the [GIF] messages from section 12.5")
for _gif in _gifs:
    show_gif(_gif)